# PyTorch 기반 RNN Seq2Seq 번역기

이 노트북은 Seq2Seq 번역기 코드를 **PyTorch 코드**로 변환한 버전입니다.

핵심 흐름은 다음과 같습니다.

- Seq2Seq는 입력 문장을 인코더가 압축 정보로 변환하고, 디코더가 그 정보를 바탕으로 출력 문장을 생성하는 구조입니다.
- 영어-프랑스어 병렬 말뭉치(`fra.txt`)를 사용합니다.
- 입력 문장에는 인코더 입력을 만들고, 출력 문장에는 `<sos>`와 `<eos>` 토큰을 추가합니다.
- 단어 집합 생성, 정수 인코딩, 패딩, 데이터 분리, LSTM 기반 인코더/디코더 모델 학습, 번역 결과 출력 순서로 실습을 진행합니다.

> 이 노트북은 PyTorch `nn.Embedding`, `nn.LSTM`, `nn.Linear`, `nn.CrossEntropyLoss`를 사용합니다.

## 1. 패키지 불러오기

Seq2Seq 실습을 위해 데이터 파일을 읽고, 문장을 정수 시퀀스로 바꾸고, 인코더와 디코더를 구성하는 흐름을 설명합니다.  
아래 코드는 PyTorch 학습에 필요한 패키지를 한 번에 불러옵니다.

In [1]:
# 운영체제의 파일 경로 처리, 폴더 생성, 파일 존재 여부 확인 등에 사용하는 표준 라이브러리입니다.
import os

# 정규표현식을 사용하여 문장 안의 특수문자, 구두점, 공백 등을 정리하기 위한 표준 라이브러리입니다.
import re

# zip 파일 압축 해제에 사용하는 표준 라이브러리입니다.
import zipfile

# URL에서 데이터 파일을 다운로드하기 위해 사용하는 표준 라이브러리입니다.
import urllib.request

# 유니코드 문자의 악센트를 제거하기 위해 사용하는 표준 라이브러리입니다.
import unicodedata

# 난수 고정을 통해 매번 비슷한 학습 결과가 나오도록 설정하기 위해 사용하는 표준 라이브러리입니다.
import random

# 데이터프레임 형태로 데이터를 확인하고 처리하기 위해 사용하는 외부 라이브러리입니다.
import pandas as pd

# 수치 계산과 배열 처리를 위해 사용하는 외부 라이브러리입니다.
import numpy as np

# PyTorch의 핵심 패키지로, 텐서 생성과 GPU 연산을 담당합니다.
import torch

# PyTorch에서 신경망 계층, 손실 함수 등을 만들기 위해 사용하는 모듈입니다.
import torch.nn as nn

# PyTorch에서 Adam 같은 최적화 알고리즘을 사용하기 위한 모듈입니다.
import torch.optim as optim

# Dataset과 DataLoader를 사용하여 데이터를 미니배치 단위로 공급하기 위한 모듈입니다.
from torch.utils.data import Dataset, DataLoader

## 2. 실행 환경과 난수 고정

딥러닝 모델은 난수 초기화, 데이터 섞기, GPU 연산 방식에 따라 결과가 조금씩 달라질 수 있습니다.  
아래 코드는 실습 결과를 최대한 일정하게 유지하기 위해 난수를 고정하고, GPU가 있으면 GPU를 사용하도록 설정합니다.

In [2]:
# Python random 모듈에서 사용하는 난수 시드를 고정합니다.
random.seed(42)

# NumPy에서 사용하는 난수 시드를 고정합니다.
np.random.seed(42)

# PyTorch CPU 연산에서 사용하는 난수 시드를 고정합니다.
torch.manual_seed(42)

# CUDA GPU가 사용 가능한 경우 GPU 연산의 난수 시드도 고정합니다.
torch.cuda.manual_seed_all(42)

# CUDA GPU가 있으면 'cuda'를 사용하고, 없으면 'cpu'를 사용하도록 장치를 선택합니다.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 현재 학습에 사용할 장치를 출력합니다.
print("사용 장치:", device)

사용 장치: cpu


## 3. 영어-프랑스어 번역 데이터 다운로드

 `manythings.org/anki/fra-eng.zip`에서 영어-프랑스어 병렬 데이터를 가져오는 과정을 설명합니다.  
아래 코드는 `fra.txt` 파일이 없으면 자동으로 다운로드하고 압축을 해제합니다. 인터넷 연결이 차단된 환경에서도 구조 확인이 가능하도록 작은 예제 데이터를 생성하는 예외 처리도 포함했습니다.

In [5]:
# 데이터 파일을 저장할 폴더 이름을 지정합니다.
DATA_DIR = "data"

# 영어-프랑스어 병렬 데이터가 들어 있는 zip 파일 URL을 지정합니다.
DATA_URL = "http://www.manythings.org/anki/fra-eng.zip"

# 다운로드된 zip 파일이 저장될 경로를 지정합니다.
ZIP_PATH = os.path.join(DATA_DIR, "fra-eng.zip")

# 압축 해제 후 사용할 실제 텍스트 파일 경로를 지정합니다.
FRA_TXT_PATH = os.path.join(DATA_DIR, "fra.txt")

# data 폴더가 없으면 새로 생성합니다.
os.makedirs(DATA_DIR, exist_ok=True)

# fra.txt 파일이 아직 존재하지 않는 경우에만 다운로드를 시도합니다.
if not os.path.exists(FRA_TXT_PATH):
    try:
        # 데이터 다운로드 시작 메시지를 출력합니다.
        print("fra-eng.zip 다운로드를 시작합니다.")

        # 지정한 URL에서 zip 파일을 다운로드하여 ZIP_PATH에 저장합니다.
        urllib.request.urlretrieve(DATA_URL, ZIP_PATH)

        # 다운로드한 zip 파일을 읽기 모드로 엽니다.
        with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
            # zip 파일 안의 모든 파일을 DATA_DIR 폴더에 압축 해제합니다.
            zip_ref.extractall(DATA_DIR)

        # 다운로드와 압축 해제가 끝났음을 출력합니다.
        print("데이터 다운로드 및 압축 해제가 완료되었습니다.")

    except Exception as e:
        # 인터넷 연결 또는 다운로드 실패 시 오류 내용을 출력합니다.
        print("다운로드 실패:", e)

        if os.path.exists(ZIP_PATH):
          # 다운로드한 zip 파일을 읽기 모드로 엽니다.
          with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
              # zip 파일 안의 모든 파일을 DATA_DIR 폴더에 압축 해제합니다.
              zip_ref.extractall(DATA_DIR)

          # 다운로드와 압축 해제가 끝났음을 출력합니다.
          print("저장된 데이터 압축 해제가 완료되었습니다.")

        else:
          # 실습 구조 확인용 소규모 예제 데이터를 생성합니다.
          sample_lines = [
              "Go.\tVa !\tCC-BY 2.0",
              "Hi.\tSalut !\tCC-BY 2.0",
              "Run!\tCours !\tCC-BY 2.0",
              "I love you.\tJe t'aime.\tCC-BY 2.0",
              "Thank you.\tMerci.\tCC-BY 2.0",
              "I am hungry.\tJ'ai faim.\tCC-BY 2.0",
              "Good morning.\tBonjour.\tCC-BY 2.0",
          ]

          # 예제 데이터를 fra.txt 형식으로 저장합니다.
          with open(FRA_TXT_PATH, "w", encoding="utf-8") as f:
              # 각 예제 문장을 한 줄씩 파일에 기록합니다.
              f.write("\n".join(sample_lines))

          # 예제 데이터 생성 완료 메시지를 출력합니다.
          print("실습용 예제 fra.txt 파일을 생성했습니다.")
else:
    # fra.txt가 이미 있으면 다시 다운로드하지 않습니다.
    print("기존 fra.txt 파일을 사용합니다.")

fra-eng.zip 다운로드를 시작합니다.
다운로드 실패: HTTP Error 406: Not Acceptable
데이터 다운로드 및 압축 해제가 완료되었습니다.


## 4. 데이터 읽기

영어 문장과 프랑스어 문장이 탭(`	`)으로 구분된 파일을 읽고, 필요한 컬럼만 사용하는 흐름을 설명합니다.  
아래 코드는 원본 파일에서 `src`에는 영어 문장, `tar`에는 프랑스어 문장을 저장합니다.

In [6]:
# fra.txt 파일을 탭 구분자로 읽습니다.
lines = pd.read_csv(
    FRA_TXT_PATH,              # 읽을 파일 경로입니다.
    names=["src", "tar", "lic"], # 원본 파일은 영어, 프랑스어, 라이선스 컬럼으로 구성됩니다.
    sep="\t",                 # 컬럼이 탭 문자로 구분되어 있음을 지정합니다.
    encoding="utf-8"           # 프랑스어 악센트 문자가 깨지지 않도록 UTF-8로 읽습니다.
)

# 번역 모델 학습에는 라이선스 컬럼이 필요하지 않으므로 삭제합니다.
lines = lines[["src", "tar"]]

# 전체 데이터 개수를 확인합니다.
print("전체 샘플 개수:", len(lines))

# 데이터가 너무 많으면 실습 시간이 오래 걸리므로 일부만 사용합니다.
NUM_SAMPLES = min(5000, len(lines))

# 앞에서부터 NUM_SAMPLES개만 선택한 뒤 복사본을 만듭니다.
lines = lines.iloc[:NUM_SAMPLES].copy()

# 데이터 일부를 확인합니다.
lines.head()

전체 샘플 개수: 240521


,src,tar
0,Go.,Va !
1,Go.,Marche.
2,Go.,En route !
3,Go.,Bouge !
4,Hi.,Salut !


## 5. 문장 전처리 함수 작성

단어 레벨 번역기를 만들기 위해 입력 시퀀스에는 `<sos>`, 출력 시퀀스에는 `<eos>`를 추가한다고 설명합니다.  
그 전에 문장을 소문자로 바꾸고, 악센트를 제거하고, 구두점 앞뒤에 공백을 넣어 토큰화가 잘 되도록 전처리합니다.

In [7]:
# 프랑스어 악센트 문자를 일반 알파벳으로 바꾸기 위한 함수입니다.
def to_ascii(text):
    # unicodedata.normalize('NFD', text)는 악센트가 붙은 문자를 기본 문자와 악센트 기호로 분리합니다.
    normalized_text = unicodedata.normalize("NFD", text)

    # 유니코드 범주가 'Mn'인 문자는 악센트 표시이므로 제외하고 나머지만 연결합니다.
    ascii_text = "".join(char for char in normalized_text if unicodedata.category(char) != "Mn")

    # 악센트가 제거된 문자열을 반환합니다.
    return ascii_text

# 번역 모델에 넣기 전 문장을 정리하는 함수입니다.
def preprocess_sentence(sentence):
    # 입력 문장을 문자열로 변환하여 혹시 모를 숫자형/결측형 문제를 줄입니다.
    sentence = str(sentence)

    # 문장을 소문자로 변환하고 프랑스어 악센트를 제거합니다.
    sentence = to_ascii(sentence.lower().strip())

    # 마침표, 물음표, 느낌표, 쉼표 앞뒤에 공백을 넣어 구두점을 독립 토큰으로 분리합니다.
    sentence = re.sub(r"([?.!,¿])", r" \1 ", sentence)

    # 영어 알파벳과 주요 구두점 외의 문자는 공백으로 바꿉니다.
    sentence = re.sub(r"[^a-zA-Z?.!,¿]+", " ", sentence)

    # 여러 개의 공백을 하나의 공백으로 줄입니다.
    sentence = re.sub(r"\s+", " ", sentence)

    # 앞뒤 공백을 제거한 문장을 반환합니다.
    return sentence.strip()

# 전처리 결과를 확인하기 위한 예제 영어 문장입니다.
en_sent = "Have you had dinner?"

# 전처리 결과를 확인하기 위한 예제 프랑스어 문장입니다.
fr_sent = "Avez-vous déjà dîné?"

# 영어 문장의 전처리 전후를 출력합니다.
print("전처리 전 영어 문장:", en_sent)
print("전처리 후 영어 문장:", preprocess_sentence(en_sent))

# 프랑스어 문장의 전처리 전후를 출력합니다.
print("전처리 전 프랑스어 문장:", fr_sent)
print("전처리 후 프랑스어 문장:", preprocess_sentence(fr_sent))

전처리 전 영어 문장: Have you had dinner?
전처리 후 영어 문장: have you had dinner ?
전처리 전 프랑스어 문장: Avez-vous déjà dîné?
전처리 후 프랑스어 문장: avez vous deja dine ?


## 6. 인코더 입력, 디코더 입력, 디코더 정답 생성

Seq2Seq 학습에서는 세 가지 데이터가 필요합니다.

- 인코더 입력: 영어 문장
- 디코더 입력: `<sos>`로 시작하는 프랑스어 문장
- 디코더 정답: `<eos>`로 끝나는 프랑스어 문장

훈련 중에는 디코더가 이전 정답 토큰을 입력으로 받아 다음 토큰을 예측하는 방식, 즉 teacher forcing 구조를 사용합니다.

In [8]:
# 영어 인코더 입력 문장을 저장할 리스트를 생성합니다.
encoder_texts = []

# 프랑스어 디코더 입력 문장을 저장할 리스트를 생성합니다.
decoder_input_texts = []

# 프랑스어 디코더 정답 문장을 저장할 리스트를 생성합니다.
decoder_target_texts = []

# 데이터프레임의 각 행을 순서대로 반복합니다.
for _, row in lines.iterrows():
    # 영어 원문을 전처리하고 공백 기준으로 단어 토큰 리스트를 만듭니다.
    src_tokens = preprocess_sentence(row["src"]).split()

    # 프랑스어 번역문을 전처리하고 공백 기준으로 단어 토큰 리스트를 만듭니다.
    tar_tokens = preprocess_sentence(row["tar"]).split()

    # 빈 문장은 학습에 사용할 수 없으므로 건너뜁니다.
    if len(src_tokens) == 0 or len(tar_tokens) == 0:
        # 현재 반복을 종료하고 다음 행으로 이동합니다.
        continue

    # 인코더 입력에는 영어 토큰 리스트만 넣습니다.
    encoder_texts.append(src_tokens)

    # 디코더 입력에는 문장 시작 토큰 <sos>를 앞에 붙입니다.
    decoder_input_texts.append(["<sos>"] + tar_tokens)

    # 디코더 정답에는 문장 종료 토큰 <eos>를 뒤에 붙입니다.
    decoder_target_texts.append(tar_tokens + ["<eos>"])

# 만들어진 데이터 샘플을 확인합니다.
print("인코더 입력 예시:", encoder_texts[:3])
print("디코더 입력 예시:", decoder_input_texts[:3])
print("디코더 정답 예시:", decoder_target_texts[:3])

인코더 입력 예시: [['go', '.'], ['go', '.'], ['go', '.']]
디코더 입력 예시: [['<sos>', 'va', '!'], ['<sos>', 'marche', '.'], ['<sos>', 'en', 'route', '!']]
디코더 정답 예시: [['va', '!', '<eos>'], ['marche', '.', '<eos>'], ['en', 'route', '!', '<eos>']]


## 7. 단어 사전 생성

단어 집합의 크기를 정의하고, 단어에서 정수를 얻는 딕셔너리와 정수에서 단어를 얻는 딕셔너리를 만든다고 설명합니다.  
PyTorch 모델은 문자열을 직접 처리하지 못하므로 모든 단어를 정수 인덱스로 변환해야 합니다.

In [9]:
# 패딩 토큰의 정수 인덱스입니다. 길이가 짧은 문장을 채울 때 사용합니다.
PAD_TOKEN = "<pad>"

# 알 수 없는 단어를 처리하기 위한 토큰입니다.
UNK_TOKEN = "<unk>"

# 문장 시작을 나타내는 토큰입니다.
SOS_TOKEN = "<sos>"

# 문장 끝을 나타내는 토큰입니다.
EOS_TOKEN = "<eos>"

# 입력 언어와 출력 언어의 단어 사전을 만들기 위한 함수입니다.
def build_vocab(tokenized_sentences, add_special_tokens=True):
    # 중복 없이 단어를 모으기 위해 set 자료구조를 사용합니다.
    vocab = set()

    # 전체 문장 리스트를 반복합니다.
    for sentence in tokenized_sentences:
        # 한 문장 안의 단어들을 반복합니다.
        for token in sentence:
            # 현재 단어를 단어 집합에 추가합니다.
            vocab.add(token)

    # 재현 가능한 결과를 위해 단어를 사전순으로 정렬합니다.
    vocab = sorted(list(vocab))

    # 특수 토큰을 포함할 경우 기본 토큰들을 앞쪽 인덱스에 배치합니다.
    if add_special_tokens:
        # 기본 토큰 리스트를 정의합니다.
        base_tokens = [PAD_TOKEN, UNK_TOKEN, SOS_TOKEN, EOS_TOKEN]

        # 기본 토큰이 중복되지 않도록 제외한 뒤 단어 목록을 구성합니다.
        vocab = base_tokens + [token for token in vocab if token not in base_tokens]
    else:
        # 특수 토큰을 추가하지 않는 경우 패딩과 미등록 단어 토큰만 기본으로 둡니다.
        vocab = [PAD_TOKEN, UNK_TOKEN] + [token for token in vocab if token not in [PAD_TOKEN, UNK_TOKEN]]

    # 단어를 정수 인덱스로 바꾸는 딕셔너리를 생성합니다.
    word_to_index = {word: index for index, word in enumerate(vocab)}

    # 정수 인덱스를 단어로 되돌리는 딕셔너리를 생성합니다.
    index_to_word = {index: word for word, index in word_to_index.items()}

    # 두 딕셔너리를 반환합니다.
    return word_to_index, index_to_word

# 영어 입력 문장용 단어 사전을 생성합니다.
src_to_index, index_to_src = build_vocab(encoder_texts, add_special_tokens=False)

# 프랑스어 출력 문장용 단어 사전을 생성합니다.
tar_to_index, index_to_tar = build_vocab(decoder_input_texts + decoder_target_texts, add_special_tokens=True)

# 영어 단어 집합의 크기를 계산합니다.
src_vocab_size = len(src_to_index)

# 프랑스어 단어 집합의 크기를 계산합니다.
tar_vocab_size = len(tar_to_index)

# 단어 집합 크기를 출력합니다.
print("영어 단어 집합 크기:", src_vocab_size)
print("프랑스어 단어 집합 크기:", tar_vocab_size)

# 프랑스어 특수 토큰의 인덱스를 확인합니다.
print("프랑스어 특수 토큰 인덱스:", {token: tar_to_index[token] for token in [PAD_TOKEN, UNK_TOKEN, SOS_TOKEN, EOS_TOKEN]})

영어 단어 집합 크기: 1083
프랑스어 단어 집합 크기: 2173
프랑스어 특수 토큰 인덱스: {'<pad>': 0, '<unk>': 1, '<sos>': 2, '<eos>': 3}


## 8. 정수 인코딩과 패딩

문장을 정수 인코딩한 뒤, 패딩을 통해 길이를 동일하게 맞춘다고 설명합니다.  
아래 코드는 단어 토큰을 정수로 변환하고, 가장 긴 문장 길이에 맞춰 뒤쪽에 `<pad>`를 채웁니다.

In [10]:
# 단어 토큰 리스트를 정수 인덱스 리스트로 변환하는 함수입니다.
def tokens_to_indices(tokens, vocab):
    # 사전에 없는 단어는 <unk> 인덱스로 바꾸어 오류를 방지합니다.
    return [vocab.get(token, vocab[UNK_TOKEN]) for token in tokens]

# 여러 문장을 같은 길이로 맞추는 패딩 함수입니다.
def pad_sequences_torch(sequences, max_len, pad_value=0):
    # 패딩이 적용된 정수 시퀀스를 저장할 리스트입니다.
    padded_sequences = []

    # 모든 정수 시퀀스를 하나씩 반복합니다.
    for sequence in sequences:
        # 최대 길이보다 긴 문장은 뒤쪽을 잘라냅니다.
        sequence = sequence[:max_len]

        # 현재 문장 길이가 max_len보다 짧으면 pad_value를 뒤에 추가합니다.
        padded_sequence = sequence + [pad_value] * (max_len - len(sequence))

        # 패딩이 완료된 문장을 리스트에 저장합니다.
        padded_sequences.append(padded_sequence)

    # Python 리스트를 PyTorch LongTensor로 변환하여 반환합니다.
    return torch.tensor(padded_sequences, dtype=torch.long)

# 영어 문장들을 정수 시퀀스로 변환합니다.
encoder_sequences = [tokens_to_indices(sentence, src_to_index) for sentence in encoder_texts]

# 프랑스어 디코더 입력 문장들을 정수 시퀀스로 변환합니다.
decoder_input_sequences = [tokens_to_indices(sentence, tar_to_index) for sentence in decoder_input_texts]

# 프랑스어 디코더 정답 문장들을 정수 시퀀스로 변환합니다.
decoder_target_sequences = [tokens_to_indices(sentence, tar_to_index) for sentence in decoder_target_texts]

# 영어 입력 문장의 최대 길이를 계산합니다.
max_src_len = max(len(sequence) for sequence in encoder_sequences)

# 프랑스어 출력 문장의 최대 길이를 계산합니다.
max_tar_len = max(len(sequence) for sequence in decoder_input_sequences)

# 영어 입력 시퀀스를 같은 길이로 패딩합니다.
encoder_input = pad_sequences_torch(encoder_sequences, max_src_len, pad_value=src_to_index[PAD_TOKEN])

# 프랑스어 디코더 입력 시퀀스를 같은 길이로 패딩합니다.
decoder_input = pad_sequences_torch(decoder_input_sequences, max_tar_len, pad_value=tar_to_index[PAD_TOKEN])

# 프랑스어 디코더 정답 시퀀스를 같은 길이로 패딩합니다.
decoder_target = pad_sequences_torch(decoder_target_sequences, max_tar_len, pad_value=tar_to_index[PAD_TOKEN])

# 텐서 크기를 출력합니다.
print("인코더 입력 크기:", encoder_input.shape)
print("디코더 입력 크기:", decoder_input.shape)
print("디코더 정답 크기:", decoder_target.shape)

인코더 입력 크기: torch.Size([5000, 5])
디코더 입력 크기: torch.Size([5000, 13])
디코더 정답 크기: torch.Size([5000, 13])


## 9. 데이터 섞기와 훈련/검증 분리

테스트 데이터 분리 전 데이터를 섞고, 훈련 데이터의 일부를 평가용으로 나누는 과정을 설명합니다.  
아래 코드는 전체 데이터를 무작위로 섞은 뒤 90%는 훈련, 10%는 검증 데이터로 사용합니다.

In [11]:
# 전체 샘플 개수를 확인합니다.
num_data = encoder_input.size(0)

# 전체 데이터 인덱스를 생성합니다.
indices = torch.randperm(num_data)

# 같은 순서로 인코더 입력, 디코더 입력, 디코더 정답을 섞습니다.
encoder_input = encoder_input[indices]
decoder_input = decoder_input[indices]
decoder_target = decoder_target[indices]

# 검증 데이터 개수를 전체의 10%로 설정합니다.
num_val = max(1, int(num_data * 0.1))

# 훈련용 인코더 입력 데이터를 분리합니다.
encoder_input_train = encoder_input[:-num_val]

# 훈련용 디코더 입력 데이터를 분리합니다.
decoder_input_train = decoder_input[:-num_val]

# 훈련용 디코더 정답 데이터를 분리합니다.
decoder_target_train = decoder_target[:-num_val]

# 검증용 인코더 입력 데이터를 분리합니다.
encoder_input_val = encoder_input[-num_val:]

# 검증용 디코더 입력 데이터를 분리합니다.
decoder_input_val = decoder_input[-num_val:]

# 검증용 디코더 정답 데이터를 분리합니다.
decoder_target_val = decoder_target[-num_val:]

# 분리된 데이터 크기를 출력합니다.
print("훈련 source 데이터 크기:", encoder_input_train.shape)
print("훈련 target 입력 데이터 크기:", decoder_input_train.shape)
print("훈련 target 정답 데이터 크기:", decoder_target_train.shape)
print("검증 source 데이터 크기:", encoder_input_val.shape)
print("검증 target 입력 데이터 크기:", decoder_input_val.shape)
print("검증 target 정답 데이터 크기:", decoder_target_val.shape)

훈련 source 데이터 크기: torch.Size([4500, 5])
훈련 target 입력 데이터 크기: torch.Size([4500, 13])
훈련 target 정답 데이터 크기: torch.Size([4500, 13])
검증 source 데이터 크기: torch.Size([500, 5])
검증 target 입력 데이터 크기: torch.Size([500, 13])
검증 target 정답 데이터 크기: torch.Size([500, 13])


## 10. PyTorch Dataset과 DataLoader 구성

PyTorch에서는 학습 데이터를 `Dataset`으로 감싸고, `DataLoader`를 통해 미니배치 단위로 모델에 공급합니다.  

In [12]:
# Seq2Seq 학습용 Dataset 클래스를 정의합니다.
class TranslationDataset(Dataset):
    # Dataset 객체가 생성될 때 인코더 입력, 디코더 입력, 디코더 정답을 저장합니다.
    def __init__(self, encoder_input, decoder_input, decoder_target):
        # 인코더 입력 텐서를 객체 변수로 저장합니다.
        self.encoder_input = encoder_input

        # 디코더 입력 텐서를 객체 변수로 저장합니다.
        self.decoder_input = decoder_input

        # 디코더 정답 텐서를 객체 변수로 저장합니다.
        self.decoder_target = decoder_target

    # Dataset 전체 샘플 개수를 반환합니다.
    def __len__(self):
        # 인코더 입력의 첫 번째 차원은 샘플 개수입니다.
        return self.encoder_input.size(0)

    # 특정 인덱스의 샘플 하나를 반환합니다.
    def __getitem__(self, idx):
        # 하나의 샘플은 인코더 입력, 디코더 입력, 디코더 정답으로 구성됩니다.
        return self.encoder_input[idx], self.decoder_input[idx], self.decoder_target[idx]

# 훈련 Dataset 객체를 생성합니다.
train_dataset = TranslationDataset(encoder_input_train, decoder_input_train, decoder_target_train)

# 검증 Dataset 객체를 생성합니다.
val_dataset = TranslationDataset(encoder_input_val, decoder_input_val, decoder_target_val)

# 미니배치 크기를 지정합니다.
BATCH_SIZE = 128

# 훈련 DataLoader를 생성합니다.
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# 검증 DataLoader를 생성합니다.
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

## 11. Encoder 모델 정의

인코더가 입력 문장을 읽고 마지막 은닉 상태와 셀 상태를 디코더로 전달한다고 설명합니다.  
PyTorch에서는 `nn.Embedding`으로 단어를 벡터로 바꾸고, `nn.LSTM`으로 순서 정보를 학습합니다.

In [13]:
# 인코더 클래스를 정의합니다.
class Encoder(nn.Module):
    # 인코더에 필요한 계층을 초기화합니다.
    def __init__(self, input_vocab_size, embedding_dim, hidden_units, pad_index):
        # 부모 클래스인 nn.Module의 초기화 함수를 호출합니다.
        super().__init__()

        # 정수 인덱스로 표현된 단어를 dense vector로 바꾸는 임베딩 계층입니다.
        self.embedding = nn.Embedding(
            num_embeddings=input_vocab_size,  # 입력 언어 단어 집합 크기입니다.
            embedding_dim=embedding_dim,      # 각 단어를 표현할 임베딩 벡터 차원입니다.
            padding_idx=pad_index             # 패딩 토큰은 학습에 영향을 덜 주도록 지정합니다.
        )

        # 입력 문장의 순서 정보를 처리하는 LSTM 계층입니다.
        self.lstm = nn.LSTM(
            input_size=embedding_dim,         # LSTM에 들어가는 각 시점의 입력 벡터 크기입니다.
            hidden_size=hidden_units,         # LSTM 은닉 상태의 크기입니다.
            batch_first=True                  # 입력 텐서 형태를 (batch, seq_len, feature)로 사용합니다.
        )

    # 인코더의 순전파 연산을 정의합니다.
    def forward(self, src):
        # src의 크기는 (batch_size, src_seq_len)입니다.
        embedded = self.embedding(src)

        # embedded의 크기는 (batch_size, src_seq_len, embedding_dim)입니다.
        outputs, (hidden, cell) = self.lstm(embedded)

        # outputs는 모든 시점의 출력이고, hidden과 cell은 마지막 시점의 상태입니다.
        return hidden, cell

## 12. Decoder 모델 정의

디코더는 `<sos>` 토큰부터 시작하여 매 시점마다 다음 단어를 예측합니다.  
디코더가 매 시점마다 `tar_vocab_size`개의 단어 중 하나를 선택하는 다중 클래스 분류 문제라고 설명합니다.  
PyTorch에서는 `nn.Linear` 출력층이 각 단어에 대한 점수(logits)를 출력하고, `CrossEntropyLoss`가 정답 단어와 비교합니다.

In [14]:
# 디코더 클래스를 정의합니다.
class Decoder(nn.Module):
    # 디코더에 필요한 계층을 초기화합니다.
    def __init__(self, output_vocab_size, embedding_dim, hidden_units, pad_index):
        # 부모 클래스인 nn.Module의 초기화 함수를 호출합니다.
        super().__init__()

        # 출력 언어의 단어 인덱스를 임베딩 벡터로 변환하는 계층입니다.
        self.embedding = nn.Embedding(
            num_embeddings=output_vocab_size, # 출력 언어 단어 집합 크기입니다.
            embedding_dim=embedding_dim,      # 각 단어를 표현할 임베딩 벡터 차원입니다.
            padding_idx=pad_index             # 패딩 토큰의 인덱스입니다.
        )

        # 디코더의 LSTM 계층입니다.
        self.lstm = nn.LSTM(
            input_size=embedding_dim,         # LSTM 입력 벡터 차원입니다.
            hidden_size=hidden_units,         # LSTM 은닉 상태 차원입니다.
            batch_first=True                  # 입력 텐서를 (batch, seq_len, feature) 형태로 사용합니다.
        )

        # LSTM 출력 벡터를 출력 단어 집합 크기의 점수 벡터로 변환하는 완전연결 계층입니다.
        self.fc_out = nn.Linear(hidden_units, output_vocab_size)

    # 디코더의 순전파 연산을 정의합니다.
    def forward(self, trg, hidden, cell):
        # trg의 크기는 (batch_size, trg_seq_len)입니다.
        embedded = self.embedding(trg)

        # 인코더에서 전달받은 hidden, cell을 초기 상태로 사용하여 디코더 LSTM을 실행합니다.
        outputs, (hidden, cell) = self.lstm(embedded, (hidden, cell))

        # 각 시점의 LSTM 출력을 출력 단어 집합 크기의 logits로 변환합니다.
        predictions = self.fc_out(outputs)

        # predictions의 크기는 (batch_size, trg_seq_len, output_vocab_size)입니다.
        return predictions, hidden, cell

## 13. Seq2Seq 모델 정의

Seq2Seq 모델은 인코더와 디코더를 하나로 묶은 모델입니다.  
인코더가 입력 문장을 압축한 상태를 만들고, 디코더가 그 상태를 이용해 번역문을 생성합니다.

In [15]:
# 인코더와 디코더를 결합한 Seq2Seq 모델을 정의합니다.
class Seq2Seq(nn.Module):
    # 모델 초기화 함수입니다.
    def __init__(self, encoder, decoder):
        # 부모 클래스인 nn.Module의 초기화 함수를 호출합니다.
        super().__init__()

        # 인코더 객체를 저장합니다.
        self.encoder = encoder

        # 디코더 객체를 저장합니다.
        self.decoder = decoder

    # 순전파 연산을 정의합니다.
    def forward(self, src, trg):
        # 인코더가 입력 문장을 읽고 마지막 hidden, cell 상태를 반환합니다.
        hidden, cell = self.encoder(src)

        # 디코더는 정답 문장의 이전 토큰들을 입력으로 받아 다음 토큰들을 예측합니다.
        outputs, _, _ = self.decoder(trg, hidden, cell)

        # 전체 시점의 예측 결과를 반환합니다.
        return outputs

## 14. 모델 생성, 손실 함수, 최적화 함수 설정


PyTorch에서는  `nn.CrossEntropyLoss`를 사용합니다. 이 손실 함수는 원-핫 인코딩된 정답이 아니라 정수형 클래스 라벨을 입력받습니다.

In [16]:
# 임베딩 벡터 차원을 지정합니다.
EMBEDDING_DIM = 64

# LSTM 은닉 상태 크기를 지정합니다.
HIDDEN_UNITS = 64

# 학습 반복 횟수를 지정합니다.
EPOCHS = 10

# 영어 패딩 토큰의 인덱스를 가져옵니다.
src_pad_index = src_to_index[PAD_TOKEN]

# 프랑스어 패딩 토큰의 인덱스를 가져옵니다.
tar_pad_index = tar_to_index[PAD_TOKEN]

# 인코더 객체를 생성합니다.
encoder = Encoder(src_vocab_size, EMBEDDING_DIM, HIDDEN_UNITS, src_pad_index)

# 디코더 객체를 생성합니다.
decoder = Decoder(tar_vocab_size, EMBEDDING_DIM, HIDDEN_UNITS, tar_pad_index)

# Seq2Seq 모델 객체를 생성하고 학습 장치로 이동합니다.
model = Seq2Seq(encoder, decoder).to(device)

# 패딩 위치는 손실 계산에서 제외하도록 ignore_index를 지정합니다.
criterion = nn.CrossEntropyLoss(ignore_index=tar_pad_index)

# Adam 최적화 함수를 생성합니다.
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 모델 구조를 출력합니다.
print(model)

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(1083, 64, padding_idx=0)
    (lstm): LSTM(64, 64, batch_first=True)
  )
  (decoder): Decoder(
    (embedding): Embedding(2173, 64, padding_idx=0)
    (lstm): LSTM(64, 64, batch_first=True)
    (fc_out): Linear(in_features=64, out_features=2173, bias=True)
  )
)


## 15. 학습 함수 작성

각 배치에서 모델은 `(batch_size, target_length, vocab_size)` 형태의 예측값을 만듭니다.  
`CrossEntropyLoss`는 `(batch_size * target_length, vocab_size)` 형태의 예측값과 `(batch_size * target_length)` 형태의 정답을 받으므로 텐서 모양을 바꿔 계산합니다.

In [17]:
# 한 에폭 동안 모델을 학습하는 함수를 정의합니다.
def train_one_epoch(model, data_loader, optimizer, criterion, device):
    # 모델을 학습 모드로 전환합니다.
    model.train()

    # 에폭 전체 손실을 누적할 변수를 초기화합니다.
    total_loss = 0.0

    # DataLoader에서 미니배치를 하나씩 가져옵니다.
    for src, dec_in, dec_out in data_loader:
        # 인코더 입력을 학습 장치로 이동합니다.
        src = src.to(device)

        # 디코더 입력을 학습 장치로 이동합니다.
        dec_in = dec_in.to(device)

        # 디코더 정답을 학습 장치로 이동합니다.
        dec_out = dec_out.to(device)

        # 이전 배치에서 계산된 기울기를 초기화합니다.
        optimizer.zero_grad()

        # 모델에 인코더 입력과 디코더 입력을 넣어 예측값을 계산합니다.
        predictions = model(src, dec_in)

        # predictions의 크기에서 출력 단어 집합 크기를 가져옵니다.
        output_dim = predictions.size(-1)

        # CrossEntropyLoss 입력 형식에 맞게 예측값을 2차원으로 펼칩니다.
        predictions = predictions.reshape(-1, output_dim)

        # 정답도 1차원으로 펼칩니다.
        dec_out = dec_out.reshape(-1)

        # 예측값과 정답을 비교하여 손실을 계산합니다.
        loss = criterion(predictions, dec_out)

        # 손실에 대한 역전파를 수행하여 기울기를 계산합니다.
        loss.backward()

        # 기울기 폭주를 줄이기 위해 gradient clipping을 적용합니다.
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # 계산된 기울기를 이용하여 모델 파라미터를 업데이트합니다.
        optimizer.step()

        # 현재 배치 손실을 누적합니다.
        total_loss += loss.item()

    # 전체 배치의 평균 손실을 반환합니다.
    return total_loss / len(data_loader)

# 검증 데이터를 이용해 모델 손실을 계산하는 함수를 정의합니다.
def evaluate(model, data_loader, criterion, device):
    # 모델을 평가 모드로 전환합니다.
    model.eval()

    # 검증 손실을 누적할 변수를 초기화합니다.
    total_loss = 0.0

    # 평가 중에는 기울기를 계산하지 않습니다.
    with torch.no_grad():
        # DataLoader에서 미니배치를 하나씩 가져옵니다.
        for src, dec_in, dec_out in data_loader:
            # 인코더 입력을 학습 장치로 이동합니다.
            src = src.to(device)

            # 디코더 입력을 학습 장치로 이동합니다.
            dec_in = dec_in.to(device)

            # 디코더 정답을 학습 장치로 이동합니다.
            dec_out = dec_out.to(device)

            # 모델 예측값을 계산합니다.
            predictions = model(src, dec_in)

            # 출력 단어 집합 크기를 가져옵니다.
            output_dim = predictions.size(-1)

            # 손실 함수 입력 형식에 맞게 예측값을 펼칩니다.
            predictions = predictions.reshape(-1, output_dim)

            # 손실 함수 입력 형식에 맞게 정답을 펼칩니다.
            dec_out = dec_out.reshape(-1)

            # 검증 손실을 계산합니다.
            loss = criterion(predictions, dec_out)

            # 현재 배치 손실을 누적합니다.
            total_loss += loss.item()

    # 전체 검증 배치의 평균 손실을 반환합니다.
    return total_loss / len(data_loader)

## 16. 모델 학습

PyTorch에서는 직접 `for epoch in range(...)` 반복문을 작성하여 학습과 검증을 수행합니다.

In [18]:
# 학습 이력을 저장할 리스트입니다.
history = []

# 지정한 에폭 수만큼 반복 학습합니다.
for epoch in range(1, EPOCHS + 1):
    # 한 에폭 동안 훈련 데이터를 사용하여 모델을 학습합니다.
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)

    # 검증 데이터를 사용하여 현재 모델의 손실을 계산합니다.
    val_loss = evaluate(model, val_loader, criterion, device)

    # 현재 에폭의 학습 결과를 저장합니다.
    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})

    # 현재 에폭의 학습 손실과 검증 손실을 출력합니다.
    print(f"Epoch [{epoch:02d}/{EPOCHS}] train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")

Epoch [01/10] train_loss=7.0213, val_loss=5.5969
Epoch [02/10] train_loss=4.6125, val_loss=4.2685
Epoch [03/10] train_loss=3.9929, val_loss=4.0697
Epoch [04/10] train_loss=3.8024, val_loss=3.9139
Epoch [05/10] train_loss=3.6346, val_loss=3.7684
Epoch [06/10] train_loss=3.4741, val_loss=3.6357
Epoch [07/10] train_loss=3.3234, val_loss=3.5173
Epoch [08/10] train_loss=3.1857, val_loss=3.4122
Epoch [09/10] train_loss=3.0631, val_loss=3.3165
Epoch [10/10] train_loss=2.9568, val_loss=3.2387


## 17. 번역 함수 작성

평가 단계에서는 정답 문장 전체를 디코더에 넣지 않습니다.  
첫 입력은 `<sos>`이고, 이후에는 이전 시점에서 예측한 단어를 다음 시점의 입력으로 사용합니다.  
`<eos>`가 나오면 번역 생성을 종료합니다.

In [19]:
# 정수 시퀀스를 영어 문장으로 되돌리는 함수입니다.
def seq_to_src(input_seq):
    # 복원된 단어를 저장할 리스트입니다.
    words = []

    # 정수 시퀀스의 각 인덱스를 반복합니다.
    for index in input_seq:
        # Tensor 값이면 Python 정수로 변환합니다.
        index = int(index)

        # 패딩 토큰은 출력하지 않습니다.
        if index != src_to_index[PAD_TOKEN]:
            # 정수 인덱스를 영어 단어로 변환하여 리스트에 추가합니다.
            words.append(index_to_src.get(index, UNK_TOKEN))

    # 단어 리스트를 공백으로 연결하여 문장으로 반환합니다.
    return " ".join(words)

# 정수 시퀀스를 프랑스어 문장으로 되돌리는 함수입니다.
def seq_to_tar(input_seq):
    # 복원된 단어를 저장할 리스트입니다.
    words = []

    # 정수 시퀀스의 각 인덱스를 반복합니다.
    for index in input_seq:
        # Tensor 값이면 Python 정수로 변환합니다.
        index = int(index)

        # 패딩, 시작, 종료 토큰은 사람이 읽는 번역문에서 제외합니다.
        if index not in [tar_to_index[PAD_TOKEN], tar_to_index[SOS_TOKEN], tar_to_index[EOS_TOKEN]]:
            # 정수 인덱스를 프랑스어 단어로 변환하여 리스트에 추가합니다.
            words.append(index_to_tar.get(index, UNK_TOKEN))

    # 단어 리스트를 공백으로 연결하여 문장으로 반환합니다.
    return " ".join(words)

# 새로운 영어 문장을 프랑스어로 번역하는 함수입니다.
def translate_sentence(sentence, model, max_len=30):
    # 모델을 평가 모드로 전환합니다.
    model.eval()

    # 입력 문장을 전처리하고 단어 토큰 리스트로 변환합니다.
    src_tokens = preprocess_sentence(sentence).split()

    # 입력 단어들을 정수 인덱스로 변환합니다.
    src_indices = tokens_to_indices(src_tokens, src_to_index)

    # 입력 시퀀스를 텐서로 변환하고 배치 차원을 추가합니다.
    src_tensor = torch.tensor(src_indices, dtype=torch.long).unsqueeze(0).to(device)

    # 기울기 계산 없이 번역을 수행합니다.
    with torch.no_grad():
        # 인코더에서 마지막 hidden, cell 상태를 얻습니다.
        hidden, cell = model.encoder(src_tensor)

    # 첫 디코더 입력은 <sos> 토큰입니다.
    current_token = torch.tensor([[tar_to_index[SOS_TOKEN]]], dtype=torch.long).to(device)

    # 예측된 단어를 저장할 리스트입니다.
    translated_tokens = []

    # 최대 길이만큼 반복하여 단어를 하나씩 생성합니다.
    for _ in range(max_len):
        # 기울기 계산 없이 다음 단어를 예측합니다.
        with torch.no_grad():
            # 현재 토큰과 이전 상태를 디코더에 넣어 다음 토큰의 점수를 얻습니다.
            output, hidden, cell = model.decoder(current_token, hidden, cell)

        # 마지막 시점의 출력 점수에서 가장 큰 값을 가진 단어 인덱스를 선택합니다.
        next_token = output[:, -1, :].argmax(dim=-1).item()

        # <eos>가 나오면 문장 생성이 끝났으므로 반복을 중단합니다.
        if next_token == tar_to_index[EOS_TOKEN]:
            break

        # 예측된 단어를 리스트에 추가합니다.
        translated_tokens.append(index_to_tar.get(next_token, UNK_TOKEN))

        # 방금 예측한 단어를 다음 시점의 디코더 입력으로 사용합니다.
        current_token = torch.tensor([[next_token]], dtype=torch.long).to(device)

    # 예측된 단어들을 공백으로 연결하여 번역문을 반환합니다.
    return " ".join(translated_tokens)

## 18. 학습 데이터 샘플 번역 결과 확인

학습된 번역기로 일부 입력 문장을 번역해 보며, 입력 문장, 정답 문장, 번역 문장을 비교합니다.

In [20]:
# 결과를 확인할 샘플 인덱스 목록을 지정합니다.
sample_indices = [0, 1, 2, min(10, len(encoder_input_train) - 1), min(100, len(encoder_input_train) - 1)]

# 샘플 인덱스를 하나씩 반복합니다.
for seq_index in sample_indices:
    # 현재 인덱스의 영어 입력 문장을 복원합니다.
    source_sentence = seq_to_src(encoder_input_train[seq_index])

    # 현재 인덱스의 프랑스어 정답 문장을 복원합니다.
    target_sentence = seq_to_tar(decoder_target_train[seq_index])

    # 모델을 이용하여 영어 문장을 프랑스어로 번역합니다.
    predicted_sentence = translate_sentence(source_sentence, model, max_len=max_tar_len + 5)

    # 구분선을 출력합니다.
    print("-" * 60)

    # 입력 문장을 출력합니다.
    print("입력 문장:", source_sentence)

    # 정답 문장을 출력합니다.
    print("정답 문장:", target_sentence)

    # 번역 문장을 출력합니다.
    print("번역 문장:", predicted_sentence)

------------------------------------------------------------
입력 문장: stand back !
정답 문장: reculez vous !
번역 문장: ne
------------------------------------------------------------
입력 문장: see you .
정답 문장: ciao .
번역 문장: vous etes .
------------------------------------------------------------
입력 문장: terrific !
정답 문장: super !
번역 문장: c est .
------------------------------------------------------------
입력 문장: she blushed .
정답 문장: elle devenait rouge .
번역 문장: tom est .
------------------------------------------------------------
입력 문장: leave it .
정답 문장: laissez tomber !
번역 문장: c est .


## 19. 모델과 사전 저장

학습된 모델을 나중에 다시 사용하려면 모델 가중치와 단어 사전을 함께 저장해야 합니다.  
단어 사전이 없으면 정수 인덱스가 어떤 단어를 의미하는지 알 수 없기 때문입니다.

In [21]:
# 모델 저장 폴더를 지정합니다.
SAVE_DIR = "saved_model"

# 저장 폴더가 없으면 생성합니다.
os.makedirs(SAVE_DIR, exist_ok=True)

# 모델 가중치와 사전 정보를 하나의 체크포인트 딕셔너리로 구성합니다.
checkpoint = {
    "model_state_dict": model.state_dict(),       # 학습된 PyTorch 모델 파라미터입니다.
    "src_to_index": src_to_index,                 # 영어 단어를 정수로 바꾸는 사전입니다.
    "index_to_src": index_to_src,                 # 정수를 영어 단어로 바꾸는 사전입니다.
    "tar_to_index": tar_to_index,                 # 프랑스어 단어를 정수로 바꾸는 사전입니다.
    "index_to_tar": index_to_tar,                 # 정수를 프랑스어 단어로 바꾸는 사전입니다.
    "src_vocab_size": src_vocab_size,             # 영어 단어 집합 크기입니다.
    "tar_vocab_size": tar_vocab_size,             # 프랑스어 단어 집합 크기입니다.
    "embedding_dim": EMBEDDING_DIM,               # 임베딩 차원입니다.
    "hidden_units": HIDDEN_UNITS,                 # LSTM 은닉 상태 크기입니다.
    "max_src_len": max_src_len,                   # 영어 문장의 최대 길이입니다.
    "max_tar_len": max_tar_len                    # 프랑스어 문장의 최대 길이입니다.
}

# 체크포인트 파일 경로를 지정합니다.
checkpoint_path = os.path.join(SAVE_DIR, "seq2seq_translation_torch.pt")

# 체크포인트를 파일로 저장합니다.
torch.save(checkpoint, checkpoint_path)

# 저장 완료 메시지를 출력합니다.
print("모델 저장 완료:", checkpoint_path)

모델 저장 완료: saved_model/seq2seq_translation_torch.pt


## 20. 직접 문장 입력 후 번역하기

아래 셀에서 영어 문장을 바꾸면 학습된 모델이 프랑스어 번역문을 출력합니다.  
데이터를 적게 학습하면 번역 품질은 낮을 수 있으므로, 실제 품질을 높이려면 샘플 수와 에폭 수를 늘려야 합니다.

In [22]:
# 번역할 영어 문장을 지정합니다.
input_sentence = "I love you."

# 입력 문장을 모델로 번역합니다.
translated_sentence = translate_sentence(input_sentence, model, max_len=max_tar_len + 5)

# 원문을 출력합니다.
print("입력 문장:", input_sentence)

# 번역 결과를 출력합니다.
print("번역 결과:", translated_sentence)

입력 문장: I love you.
번역 결과: je suis .
